In [2]:
from keras.models import Sequential
from keras.layers import SimpleRNN

model = Sequential()
model.add(SimpleRNN(3, input_shape = (2, 10)))
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn (SimpleRNN)      (None, 3)                 42        
                                                                 
Total params: 42 (168.00 Byte)
Trainable params: 42 (168.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


#### 2.16.0 이후에 아래와 같은 문법 사용

In [6]:
from keras.models import Sequential
from keras.layers import SimpleRNN
import numpy as np

inputs = np.random.random((8, 2, 10)) # (배치사이즈, 타임스탭, 입력 크기)

model = Sequential()
model.add(SimpleRNN(3, return_sequences = True)) # return_sequences 매개 변수에 True를 기재하면 출력값으로 3D 텐서를 출력한다.
model(inputs)
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_4 (SimpleRNN)    (8, 2, 3)                 42        
                                                                 
Total params: 42 (168.00 Byte)
Trainable params: 42 (168.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# 바닐라 RNN을 통해 문장 생성기 만들기

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from tensorflow.keras.utils import to_categorical

In [10]:
text = '''경마장에 있는 말이 뛰고 있다.\n그의 말이 법이다\n가는 말이 고와야 오는 말이 곱다'''

In [12]:
t = Tokenizer() # 주어진 문장을 단어 토큰으로 만드는 클래스
t.fit_on_texts([text]) # 토큰을 만듬
vocab_size = len(t.word_index) + 1 # 원핫인코딩 시 인덱스 유지하고 싶어서 +1을 했다.
print('단어 집합의 크기 : %d' % vocab_size)

단어 집합의 크기 : 12


In [13]:
print(t.word_index) # 단어 빈도수가 높은 순서대로

{'말이': 1, '경마장에': 2, '있는': 3, '뛰고': 4, '있다': 5, '그의': 6, '법이다': 7, '가는': 8, '고와야': 9, '오는': 10, '곱다': 11}


In [16]:
# 문장을 정수로 변환하는 로직
# 경마장에 있는 말이 뛰고 있다. => 2 3 1 4 5
sequences = list()

for line in text.split('\n'):
    print(t.texts_to_sequences([line]))
    encoded = t.texts_to_sequences([line])[0]  
    # 학습 데이터 만들기
    for i in range(1, len(encoded)):
        sequence = encoded[: i + 1]
        sequences.append(sequence)

print('학습에 사용할 샘플의 개수 : %d' % len(sequences))

[[2, 3, 1, 4, 5]]
[[6, 1, 7]]
[[8, 1, 9, 10, 1, 11]]
학습에 사용할 샘플의 개수 : 11


In [19]:
sequences 
# 학습할 길이가 다르므로 동일하게 만들어줘야됨
# 그래서 맨 마지막을 타켓이기 때문에 앞에서 체워야 한다.

[[2, 3],
 [2, 3, 1],
 [2, 3, 1, 4],
 [2, 3, 1, 4, 5],
 [6, 1],
 [6, 1, 7],
 [8, 1],
 [8, 1, 9],
 [8, 1, 9, 10],
 [8, 1, 9, 10, 1],
 [8, 1, 9, 10, 1, 11]]

In [21]:
max_len = max(len(l) for l in sequences)
print('샘플의 최대 길이 : {}'.format(max_len))

샘플의 최대 길이 : 6


* pad_sequences(sequences, maxlen = None, dtype = 'int32', padding = 'pre', truncating = 'pre',
                  value = 0.0)
    * 인수
        * padding : 'pre' or 'post'
        * t

In [23]:
sequences = pad_sequences(sequences, maxlen = max_len, padding = 'pre')
sequences

array([[ 0,  0,  0,  0,  2,  3],
       [ 0,  0,  0,  2,  3,  1],
       [ 0,  0,  2,  3,  1,  4],
       [ 0,  2,  3,  1,  4,  5],
       [ 0,  0,  0,  0,  6,  1],
       [ 0,  0,  0,  6,  1,  7],
       [ 0,  0,  0,  0,  8,  1],
       [ 0,  0,  0,  8,  1,  9],
       [ 0,  0,  8,  1,  9, 10],
       [ 0,  8,  1,  9, 10,  1],
       [ 8,  1,  9, 10,  1, 11]])

In [26]:
# 학습 데이터와 테스트 데이터 분류

sequences = np.array(sequences)
X = sequences[:, : -1]
y = sequences[:,-1]

In [27]:
y = to_categorical(y, num_classes = vocab_size)
print(y)

[[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN

In [34]:
model = Sequential()
model.add(Embedding(vocab_size, 6)) # 단어로 구성된 벡터를 밀집벡터로 변환한다. 밀집벡터는 실수형으로 구성된다.
model.add(SimpleRNN(32))
model.add(Dense(vocab_size, activation = 'softmax'))
model.build(input_shape = (None, max_len - 1))
model.summary()

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, None, 6)           72        
                                                                 
 simple_rnn_6 (SimpleRNN)    (None, 32)                1248      
                                                                 
 dense (Dense)               (None, 12)                396       
                                                                 
Total params: 1716 (6.70 KB)
Trainable params: 1716 (6.70 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [37]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'])
model.fit(X,y, epochs=200, verbose=2)


Epoch 1/200


1/1 - 1s - loss: 2.4875 - accuracy: 0.0000e+00 - 761ms/epoch - 761ms/step
Epoch 2/200
1/1 - 0s - loss: 2.4741 - accuracy: 0.2727 - 6ms/epoch - 6ms/step
Epoch 3/200
1/1 - 0s - loss: 2.4606 - accuracy: 0.3636 - 3ms/epoch - 3ms/step
Epoch 4/200
1/1 - 0s - loss: 2.4468 - accuracy: 0.3636 - 1ms/epoch - 1ms/step
Epoch 5/200
1/1 - 0s - loss: 2.4328 - accuracy: 0.3636 - 0s/epoch - 0s/step
Epoch 6/200
1/1 - 0s - loss: 2.4184 - accuracy: 0.3636 - 2ms/epoch - 2ms/step
Epoch 7/200
1/1 - 0s - loss: 2.4034 - accuracy: 0.3636 - 3ms/epoch - 3ms/step
Epoch 8/200
1/1 - 0s - loss: 2.3878 - accuracy: 0.3636 - 4ms/epoch - 4ms/step
Epoch 9/200
1/1 - 0s - loss: 2.3714 - accuracy: 0.3636 - 4ms/epoch - 4ms/step
Epoch 10/200
1/1 - 0s - loss: 2.3543 - accuracy: 0.3636 - 4ms/epoch - 4ms/step
Epoch 11/200
1/1 - 0s - loss: 2.3364 - accuracy: 0.3636 - 2ms/epoch - 2ms/step
Epoch 12/200
1/1 - 0s - loss: 2.3175 - accuracy: 0.3636 - 4ms/epoch - 4ms/step
Epoch 13/200
1/1 - 0s - loss: 2.2976 - accuracy: 0.3

In [38]:
def sentence_generation(model, t, current_word, n):
    init_word = current_word
    sentence = ''

    for _ in range(n):
        encoded = t.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen = max_len - 1, padding = 'pre')
        temp = model.predict(encoded)
        result = np.argmax(temp, axis = 1)

        for word, index in t.word_index.items():
            if index == result:
                break

        current_word = current_word + ' ' + word
        sentence = init_word + sentence

    sentence = init_word + sentence

    return sentence

In [39]:
print(sentence_generation(model, t, '경마장에', 4))

1/1 [==============================] - 0s 12ms/step
경마장에경마장에경마장에경마장에경마장에


# LSTM을 이용하여 텍스트 생성

In [40]:
import pandas as pd
from string import punctuation
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from tensorflow.keras.utils import to_categorical

In [41]:
df = pd.read_csv('ArticlesApril2018.csv')

In [42]:
df.head()

,articleID,articleWordCount,byline,documentType,headline,keywords,multimedia,newDesk,printPage,pubDate,sectionName,snippet,source,typeOfMaterial,webURL
0,5adf6684068401528a2aa69b,781,By JOHN BRANCH,article,Former N.F.L. Cheerleaders’ Settlement Offer: ...,"['Workplace Hazards and Violations', 'Football...",68,Sports,0,2018-04-24 17:16:49,Pro Football,"“I understand that they could meet with us, pa...",The New York Times,News,https://www.nytimes.com/2018/04/24/sports/foot...
1,5adf653f068401528a2aa697,656,By LISA FRIEDMAN,article,E.P.A. to Unveil a New Rule. Its Effect: Less ...,"['Environmental Protection Agency', 'Pruitt, S...",68,Climate,0,2018-04-24 17:11:21,Unknown,The agency plans to publish a new regulation T...,The New York Times,News,https://www.nytimes.com/2018/04/24/climate/epa...
2,5adf4626068401528a2aa628,2427,By PETE WELLS,article,"The New Noma, Explained","['Restaurants', 'Noma (Copenhagen, Restaurant)...",66,Dining,0,2018-04-24 14:58:44,Unknown,What’s it like to eat at the second incarnatio...,The New York Times,News,https://www.nytimes.com/2018/04/24/dining/noma...
3,5adf40d2068401528a2aa619,626,By JULIE HIRSCHFELD DAVIS and PETER BAKER,article,Unknown,"['Macron, Emmanuel (1977- )', 'Trump, Donald J...",68,Washington,0,2018-04-24 14:35:57,Europe,President Trump welcomed President Emmanuel Ma...,The New York Times,News,https://www.nytimes.com/2018/04/24/world/europ...
4,5adf3d64068401528a2aa60f,815,By IAN AUSTEN and DAN BILEFSKY,article,Unknown,"['Toronto, Ontario, Attack (April, 2018)', 'Mu...",68,Foreign,0,2018-04-24 14:21:21,Canada,"Alek Minassian, 25, a resident of Toronto’s Ri...",The New York Times,News,https://www.nytimes.com/2018/04/24/world/canad...


In [43]:
print('열의 개수 ', len(df.columns))
print(df.columns)

열의 개수  15
Index(['articleID', 'articleWordCount', 'byline', 'documentType', 'headline',
       'keywords', 'multimedia', 'newDesk', 'printPage', 'pubDate',
       'sectionName', 'snippet', 'source', 'typeOfMaterial', 'webURL'],
      dtype='object')


In [44]:
df['headline'].isnull().values.any()

False

In [45]:
headline = []
headline.extend(list(df.headline.values))
headline[:5]

['Former N.F.L. Cheerleaders’ Settlement Offer: $1 and a Meeting With Goodell',
 'E.P.A. to Unveil a New Rule. Its Effect: Less Science in Policymaking.',
 'The New Noma, Explained',
 'Unknown',
 'Unknown']

In [46]:
print('총 샘플 수 : {}'.format(len(headline)))

총 샘플 수 : 1324


In [49]:
headline = [n for n in headline if n != 'Unknown']
print('노이즈 제거 후 총 샘플 수 {}'.format(len(headline)))

노이즈 제거 후 총 샘플 수 1214


In [50]:
headline[:5]

['Former N.F.L. Cheerleaders’ Settlement Offer: $1 and a Meeting With Goodell',
 'E.P.A. to Unveil a New Rule. Its Effect: Less Science in Policymaking.',
 'The New Noma, Explained',
 'How a Bag of Texas Dirt  Became a Times Tradition',
 'Is School a Place for Self-Expression?']

In [63]:
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [64]:
def repreprocessing(s):
    s = s.encode('utf8').decode('ascii', 'ignore')
    return ''.join(c for c in s if c not in punctuation).lower()

text = [repreprocessing(x) for x in headline]
text[:3]

['former nfl cheerleaders settlement offer 1 and a meeting with goodell',
 'epa to unveil a new rule its effect less science in policymaking',
 'the new noma explained']

In [65]:
t = Tokenizer() # 주어진 문장을 단어 토큰으로 만드는 클래스
t.fit_on_texts(text) # 토큰을 만듬
vocab_size = len(t.word_index) + 1 # 원핫인코딩 시 인덱스 유지하고 싶어서 +1을 했다.
print('단어 집합의 크기 : %d' % vocab_size)

단어 집합의 크기 : 3494


In [66]:
# 문장을 정수로 변환하는 로직
# 경마장에 있는 말이 뛰고 있다. => 2 3 1 4 5
sequences = list()

for line in text:
    print(t.texts_to_sequences([line]))
    encoded = t.texts_to_sequences([line])[0]  
    # 학습 데이터 만들기
    for i in range(1, len(encoded)):
        sequence = encoded[: i + 1]
        sequences.append(sequence)

print('학습에 사용할 샘플의 개수 : %d' % sequences[:11])

[[99, 269, 371, 1115, 582, 52, 7, 2, 372, 10, 1116]]
[[100, 3, 1117, 2, 14, 583, 24, 1118, 373, 374, 5, 1119]]
[[1, 14, 1120, 1121]]
[[15, 2, 584, 4, 215, 375, 1122, 2, 376, 585]]
[[8, 167, 2, 136, 6, 1123]]
[[1124, 1125]]
[[1126, 1127, 168, 377, 6, 2, 1128, 24, 88, 377]]
[[1129, 586, 3, 378, 13, 587, 1130, 66, 588, 589, 216, 1131]]
[[1132, 1133]]
[[30, 590, 1, 1134, 3, 1135, 217, 218, 31, 216, 372, 10, 379, 1136]]
[[5, 1137, 591, 219, 2, 67]]
[[1, 80, 380]]
[[270, 592, 1138, 25, 89, 169, 381]]
[[1, 220, 8, 36, 170]]
[[14, 593, 382, 137, 1139, 101, 42, 1140, 1141]]
[[1142]]
[[271, 594, 1143]]
[[90, 595, 1144, 10, 1145, 1146, 596, 1147]]
[[597, 1148, 2, 383, 1149, 6, 24, 1150, 138, 7, 1151, 24, 598]]
[[2, 1152, 1153]]
[[272, 380, 273, 384]]
[[13, 1, 171, 1, 116, 274, 43, 20, 599]]
[[139, 4, 1154, 385, 66, 30, 275, 386, 117]]
[[62, 1, 172, 387, 68, 3, 102, 11]]
[[276, 5, 221]]
[[15, 388, 140, 1155, 586, 9, 389, 1156]]
[[18, 222, 390, 5, 1157]]
[[47, 91, 92, 37, 1158]]
[[391, 1159, 7, 223

TypeError: %d format: a real number is required, not list

In [67]:
max_len = max(len(l) for l in sequences)
print('샘플의 최대 길이 : {}'.format(max_len))

샘플의 최대 길이 : 24


In [69]:
sequences = pad_sequences(sequences, maxlen = max_len, padding = 'pre')
sequences

array([[   0,    0,    0, ...,    0,   99,  269],
       [   0,    0,    0, ...,   99,  269,  371],
       [   0,    0,    0, ...,  269,  371, 1115],
       ...,
       [   0,    0,    0, ...,    8, 3493,  115],
       [   0,    0,    0, ..., 3493,  115,    2],
       [   0,    0,    0, ...,  115,    2, 1025]])

In [70]:
sequences = np.array(sequences)
X = sequences[:, : -1]
y = sequences[:,-1]

In [71]:
y = to_categorical(y, num_classes = vocab_size)
print(y)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [74]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [75]:
model = Sequential()
model.add(Embedding(vocab_size, 10)) # 단어로 구성된 벡터를 밀집벡터로 변환한다. 밀집벡터는 실수형으로 구성된다.
model.add(LSTM(32)) # model.add(Bidirectional ~~~)
model.add(Dense(vocab_size, activation = 'softmax'))
model.build(input_shape = (None, max_len - 1))
model.summary()

Model: "sequential_12"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, None, 10)          34940     
                                                                 
 lstm (LSTM)                 (None, 32)                5504      
                                                                 
 dense_2 (Dense)             (None, 3494)              115302    
                                                                 
Total params: 155746 (608.38 KB)
Trainable params: 155746 (608.38 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [76]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'])
model.fit(X,y, epochs=100, verbose=2)

Epoch 1/100
244/244 - 2s - loss: 7.6471 - accuracy: 0.0295 - 2s/epoch - 8ms/step
Epoch 2/100
244/244 - 1s - loss: 7.1239 - accuracy: 0.0317 - 983ms/epoch - 4ms/step
Epoch 3/100
244/244 - 1s - loss: 7.0135 - accuracy: 0.0323 - 977ms/epoch - 4ms/step
Epoch 4/100
244/244 - 1s - loss: 6.9300 - accuracy: 0.0306 - 1s/epoch - 4ms/step
Epoch 5/100
244/244 - 1s - loss: 6.8621 - accuracy: 0.0333 - 995ms/epoch - 4ms/step
Epoch 6/100
244/244 - 1s - loss: 6.8091 - accuracy: 0.0367 - 1s/epoch - 4ms/step
Epoch 7/100
244/244 - 1s - loss: 6.7679 - accuracy: 0.0410 - 985ms/epoch - 4ms/step
Epoch 8/100
244/244 - 1s - loss: 6.7312 - accuracy: 0.0425 - 976ms/epoch - 4ms/step
Epoch 9/100
244/244 - 1s - loss: 6.6971 - accuracy: 0.0436 - 988ms/epoch - 4ms/step
Epoch 10/100
244/244 - 1s - loss: 6.6639 - accuracy: 0.0450 - 978ms/epoch - 4ms/step
Epoch 11/100
244/244 - 1s - loss: 6.6332 - accuracy: 0.0460 - 971ms/epoch - 4ms/step
Epoch 12/100
244/244 - 1s - loss: 6.6022 - accuracy: 0.0500 - 992ms/epoch - 4ms/ste

In [81]:
def sentence_generation(model, tokenizer, start_word, n):
    result = [start_word]
    
    for _ in range(n):
        encoded = tokenizer.texts_to_sequences([' '.join(result)])[0]
        encoded = pad_sequences([encoded], maxlen=max_len - 1, padding='pre')
        
        y_pred = model.predict(encoded, verbose=0)
        predicted_id = np.argmax(y_pred)

        # index -> word 변환
        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                result.append(word)
                break

    return ' '.join(result)

In [82]:
print(sentence_generation(model, t, 'i', 10))

i going to the new house dies in the imitators dies
